# DSAI LAB 10 Proshita Agarwal

### PART 1

In [ ]:
# Q1: Classify a propositional formula as tautology, contradiction, or contingency.

from sympy import symbols, Not, satisfiable

def classify_formula(expr):
    """
    Returns one of: 'Tautology', 'Contradiction', or 'Contingency'
    """
    # if ~phi is unsatisfiable → tautology
    if not satisfiable(Not(expr)):
        return "Tautology"
    # if phi itself is unsatisfiable → contradiction
    elif not satisfiable(expr):
        return "Contradiction"
    else:
        return "Contingency"

# Example test
p, q = symbols('p q')
expr1 = p | ~p                      # tautology
expr2 = p & ~p                      # contradiction
expr3 = (p >> q)                    # contingency
print(classify_formula(expr1))
print(classify_formula(expr2))
print(classify_formula(expr3))

Tautology
Contradiction
Contingency


In [3]:
# Q2: Check whether two formulas are logically equivalent.

from sympy import Equivalent

def logically_equivalent(expr1, expr2):
    
    #Returns True if (expr1 ↔ expr2) is a tautology.
    
    return not satisfiable(Not(Equivalent(expr1, expr2)))

# Example test
p, q = symbols('p q')
a = ~(p & q)
b = (~p) | (~q)
print(logically_equivalent(a, b))  # should be True (De Morgan)

True


In [5]:
# Q3 (fixed): Convert a formula to CNF and compute clause statistics.

from sympy import symbols, And, Or
from sympy.logic.boolalg import to_cnf

def cnf_stats(expr):
    cnf_expr = to_cnf(expr, simplify=True)
    
    # break CNF into clauses
    clauses = list(cnf_expr.args) if isinstance(cnf_expr, And) else [cnf_expr]
    
    num_clauses = len(clauses)
    clause_lengths = []
    unit_clauses = []

    for c in clauses:
        lits = list(c.args) if isinstance(c, Or) else [c]
        clause_lengths.append(len(lits))
        if len(lits) == 1:
            unit_clauses.append(lits[0])
    
    avg_len = sum(clause_lengths) / num_clauses if num_clauses else 0
    
    return {
        "CNF": cnf_expr,
        "num_clauses": num_clauses,
        "avg_clause_length": avg_len,
        "unit_clauses": unit_clauses
    }

# Example test
p, q, r = symbols('p q r')
expr = (p >> q) & (q >> r)
print(cnf_stats(expr))

{'CNF': (q | ~p) & (r | ~q), 'num_clauses': 2, 'avg_clause_length': 2.0, 'unit_clauses': []}


In [ ]:
# Q4: Check whether Γ entails ψ, i.e. Γ ⊨ ψ.
# Also produce an approximate minimal unsat core (greedy deletion).
# took help from gpt for this one
def entails(premises, conclusion):
    """
    Returns (entails?, approx_core)
    - premises: list of sympy expressions
    - conclusion: sympy expression
    """
    from copy import deepcopy

    # Γ ∧ ¬ψ is unsatisfiable → entailment holds
    conjunction = And(*premises, Not(conclusion))
    ent = not satisfiable(conjunction)

    # greedy approximate minimal UNSAT core
    core = deepcopy(premises)
    if ent:
        for f in premises:
            temp = [x for x in core if x != f]
            if not satisfiable(And(*temp, Not(conclusion))):
                core = temp
    return ent, core

# Example
p, q = symbols('p q')
Gamma = [p >> q, p]
psi = q
print(entails(Gamma, psi))

(True, [Implies(p, q), p])


In [ ]:
# Q5 (fixed): Compute minimal DNF and compare with CNF; show prime implicants.

from sympy import symbols, And, Or
from sympy.logic.boolalg import simplify_logic

def minimal_dnf(expr):
    
    """
    Returns the minimal DNF and a list of prime implicants.
    Works across SymPy versions.
    """
    dnf_expr = simplify_logic(expr, form='dnf', force=True)

    # Extract prime implicants safely
    if isinstance(dnf_expr, Or):
        primes = list(dnf_expr.args)
    else:
        primes = [dnf_expr]

    return dnf_expr, primes

# Example 1: Majority-of-three function
p, q, r = symbols('p q r')
majority = (p & q) | (p & r) | (q & r)
dnf_form, primes = minimal_dnf(majority)

print("Minimal DNF:", dnf_form)
print("Prime implicants:", primes)

# Compare with CNF
from sympy.logic.boolalg import simplify_logic
cnf_form = simplify_logic(majority, form='cnf', force=True)
print("CNF:", cnf_form)

Minimal DNF: (p & q) | (p & r) | (q & r)
Prime implicants: [p & q, p & r, q & r]
CNF: (p | q) & (p | r) & (q | r)


### PART 2

In [9]:
# Q1: Verify a quantified claim using finite model checking.
# Example domain: Students, Courses, Assignments

Students = {"Alice", "Bob", "Charlie"}
Courses = {"AI", "ML"}
Assignments = {("Alice", "AI"), ("Bob", "ML"), ("Charlie", "AI")}

# Claim: ∀s ∈ Students, ∃c ∈ Courses such that (s, c) ∈ Assignments
def forall_exists(domain1, domain2, relation):
    for s in domain1:
        if not any((s, c) in relation for c in domain2):
            return False, s  # counterexample
    return True, None

holds, witness = forall_exists(Students, Courses, Assignments)
print("Claim holds?" , holds)
if not holds:
    print("Counterexample student:", witness)

Claim holds? True


In [10]:
# Q2: Check if a dataset contains contradictory facts.
# Example: likes(A,B) and not likes(A,B)

likes = {("Alice", "Pizza"), ("Bob", "Burger")}
dislikes = {("Alice", "Pizza")}  # contradiction here

def inconsistent(pred, neg_pred):
    for fact in pred:
        if fact in neg_pred:
            return True, fact
    return False, None

flag, witness = inconsistent(likes, dislikes)
print("Inconsistent?", flag)
if flag:
    print("Contradiction at:", witness)

Inconsistent? True
Contradiction at: ('Alice', 'Pizza')


In [11]:
# Q3: Generate Ancestor relation from Parent using forward chaining.

Parent = {("Alice", "Bob"), ("Bob", "Charlie")}
Ancestor = set(Parent)  # every parent is an ancestor

# Closure rule: if (x,y) and (y,z), then (x,z)
changed = True
while changed:
    changed = False
    new_pairs = set()
    for x, y in Ancestor:
        for a, b in Ancestor:
            if y == a and (x, b) not in Ancestor:
                new_pairs.add((x, b))
    if new_pairs:
        Ancestor |= new_pairs
        changed = True

print("Ancestor relation:", Ancestor)

# Individuals with no ancestors
all_people = {p for pair in Ancestor for p in pair}
no_ancestors = {p for p in all_people if all(p != b for _, b in Ancestor)}
print("People with no ancestors:", no_ancestors)

Ancestor relation: {('Bob', 'Charlie'), ('Alice', 'Charlie'), ('Alice', 'Bob')}
People with no ancestors: {'Alice'}


In [12]:
# Q4: Build datasets to show ∀s ∃c Enrolled(s,c) is true
# but ∃c ∀s Enrolled(s,c) is false.

Students = {"A", "B", "C"}
Courses = {"AI", "ML"}

Enrolled = {("A", "AI"), ("B", "ML"), ("C", "AI")}  # every student has at least one course

def forall_exists(domain1, domain2, relation):
    for s in domain1:
        if not any((s, c) in relation for c in domain2):
            return False, s
    return True, None

def exists_forall(domain1, domain2, relation):
    for c in domain2:
        if all((s, c) in relation for s in domain1):
            return True, c
    return False, None

f1, w1 = forall_exists(Students, Courses, Enrolled)
f2, w2 = exists_forall(Students, Courses, Enrolled)

print("∀s ∃c Enrolled(s,c):", f1)
print("∃c ∀s Enrolled(s,c):", f2)
if not f2:
    print("No single course has all students.")

∀s ∃c Enrolled(s,c): True
∃c ∀s Enrolled(s,c): False
No single course has all students.


In [13]:
# Q5: Check rule preservation and find minimal counterexample if violated.
# Rule: If student takes AI_Course, then they must have a project.

AI_Course = {"Alice", "Bob", "Charlie"}
HasProject = {"Alice", "Charlie"}  # Bob missing → violates rule

def check_rule(premise_set, implied_set):
    for x in premise_set:
        if x not in implied_set:
            return False, x
    return True, None

ok, witness = check_rule(AI_Course, HasProject)
print("Rule holds?", ok)
if not ok:
    print("Counterexample:", witness)
    print("Minimal data fix → add:", witness, "to HasProject")

Rule holds? False
Counterexample: Bob
Minimal data fix → add: Bob to HasProject


### PART 3

In [14]:
# Q1: Verify environment setup and library versions.

import sys, platform
import sympy

print("Python version:", sys.version)
print("Platform:", platform.system(), platform.release())
print("SymPy version:", sympy.__version__)

Python version: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Platform: Windows 10
SymPy version: 1.14.0


In [15]:
# Q2: Generate a truth table for any propositional formula.

from sympy import symbols
from sympy.logic.boolalg import truth_table

def show_truth_table(expr):
    print(f"Truth Table for: {expr}\n")
    for row in truth_table(expr, expr.free_symbols):
        vals, result = row
        print({str(s): v for s, v in zip(expr.free_symbols, vals)}, "=>", result)

# Example
p, q = symbols('p q')
show_truth_table(p >> q)

Truth Table for: Implies(p, q)

{'q': 0, 'p': 0} => True
{'q': 0, 'p': 1} => False
{'q': 1, 'p': 0} => True
{'q': 1, 'p': 1} => True


In [19]:
# Run this in a notebook cell to install tabulate (will work if environment allows pip installs)
!pip install tabulate

In [20]:
# Q3: Nicely format CNF statistics (uses the cnf_stats function from Part 1).

from tabulate import tabulate

def display_cnf_table(expr_list):
    rows = []
    for expr in expr_list:
        stats = cnf_stats(expr)
        rows.append([
            str(expr),
            str(stats['CNF']),
            stats['num_clauses'],
            round(stats['avg_clause_length'], 2),
            ", ".join(str(u) for u in stats['unit_clauses']) or "-"
        ])
    print(tabulate(rows, headers=["Formula", "CNF", "#Clauses", "AvgLen", "Unit Clauses"]))

# Example
p, q, r = symbols('p q r')
exprs = [(p >> q), (q >> r), (p & (q >> r))]
display_cnf_table(exprs)

Formula              CNF             #Clauses    AvgLen  Unit Clauses
-------------------  ------------  ----------  --------  --------------
Implies(p, q)        q | ~p                 1       2    -
Implies(q, r)        r | ~q                 1       2    -
p & (Implies(q, r))  p & (r | ~q)           2       1.5  p


In [17]:
# Q4: Generic helpers for quantified statements (reuse across labs).

def ForAll(domain, predicate):
    """
    Returns True if predicate(x) is True for every x in domain.
    """
    return all(predicate(x) for x in domain)

def Exists(domain, predicate):
    """
    Returns True if predicate(x) is True for at least one x in domain.
    """
    return any(predicate(x) for x in domain)

# Example
domain = {1, 2, 3, 4}
print("ForAll even?", ForAll(domain, lambda x: x % 2 == 0))
print("Exists even?", Exists(domain, lambda x: x % 2 == 0))

ForAll even? False
Exists even? True


In [18]:
# Q5: Small demo integrating propositional + predicate logic utilities.

p, q = symbols('p q')
expr = (p | ~q) >> (~p | q)

print("Classification:", classify_formula(expr))
print("CNF stats:", cnf_stats(expr))

# Using predicate logic example
Students = {"A", "B"}
Courses = {"AI"}
Enrolled = {("A", "AI")}  # B not enrolled → violates ∀s∃c

ok, counter = forall_exists(Students, Courses, Enrolled)
print("∀s∃c Enrolled(s,c)?", ok)
if not ok:
    print("Counterexample:", counter)

Classification: Contingency
CNF stats: {'CNF': q | ~p, 'num_clauses': 1, 'avg_clause_length': 2.0, 'unit_clauses': []}
∀s∃c Enrolled(s,c)? False
Counterexample: B


### FINAL REPORT
This lab focused on understanding and implementing both propositional and predicate logic using Python.
It gave me a chance to work with symbolic reasoning, truth evaluation, and finite model checking in a very hands-on way.

Part 1 - Propositional Logic

In the first part, I implemented functions to classify logical formulas as tautologies, contradictions, or contingencies, and to test logical equivalence between expressions.
Using SymPy, I also converted expressions to CNF form, calculated clause statistics, and explored entailment checking through satisfiability testing.
Finally, I generated minimal DNFs and extracted prime implicants, which helped me understand how logical simplification and representation size affect reasoning efficiency.

Part 2 - Predicate Logic

The second part moved to reasoning over finite models.
I built small datasets to represent relations like Enrolled, Parent, and Ancestor, then implemented custom quantifier checks (ForAll, Exists) to verify logical statements over these datasets.
I also detected inconsistencies, simulated inference using forward chaining, and compared ∀∃ vs ∃∀ quantifier scopes, which made the difference between logical formulations much clearer.

Part 3 - Tools and Environment

The last part ensured that my Python environment was properly set up and that all helper functions worked together.
I verified library versions, built a small truth-table generator, formatted CNF statistics neatly, and confirmed that both the propositional and predicate logic modules integrate smoothly.

Conclusion

Overall, this lab deepened my understanding of how formal logic can be represented and reasoned with computationally.
Translating theoretical concepts like entailment and quantification into actual Python code was the most valuable takeaway.
The step-by-step structure of the lab made complex ideas much easier to grasp, and by the end I felt confident using logic programming tools to verify, infer, and simplify statements in both propositional and predicate domains.